# اجرای پژوهشی پایان‌نامه در Google Colab

ابتدا README_FA و CONFORMANCE_FA را بخوانید. آموزش و آزمون با دادهٔ واقعی انجام می‌شود؛ خروجی ساختگی نداریم. هفت پارامتر معادلهٔ اصلی نامشخص و دستگاه پیوست متفاوت و مستعد واگرایی است. هیچ عدد فصل چهارم از پیش تأیید نشده است. این بسته برای دادهٔ شناسایی‌پذیر بیمار یا استفادهٔ بالینی نیست.

In [15]:
%pip install -q --upgrade kagglehub

## ۱. بارگذاری ZIP بسته
فایل `medical_thesis_colab.zip` را انتخاب کنید؛ این فایل فقط شامل کد و مستندات است. نوت‌بوک قدیمی اجرا نمی‌شود.

In [1]:
from pathlib import Path
import os, sys, json, zipfile, subprocess
from google.colab import files
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.endswith('.zip')]
if len(archives) != 1: raise ValueError('دقیقاً ZIP بسته را انتخاب کنید')
ROOT = Path('/content/thesis_toolkit')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as z:
    for name in z.namelist():
        if not (ROOT/name).resolve().is_relative_to(ROOT.resolve()): raise ValueError('Unsafe zip')
    z.extractall(ROOT)
PACKAGE = ROOT/'medsec_colab'
os.chdir(PACKAGE)
sys.path.insert(0, str(PACKAGE))
print('مسیر بسته:', PACKAGE)

Saving medsec_colab.zip to medsec_colab.zip
مسیر بسته: /content/thesis_toolkit/medsec_colab


## ۲. نصب و آزمون نرم‌افزار
آزمون‌های خودکار از دادهٔ کوچک مصنوعی **فقط برای بررسی نرم‌افزار** استفاده می‌کنند؛ آن‌ها آزمایش پزشکی یا بازتولید نتایج نیستند.

In [2]:
subprocess.run([sys.executable, 'scripts/install_colab.py'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', 'scripts/tests', '-q'], check=True)
from scripts.artifacts import environment
print(json.dumps(environment(), indent=2, ensure_ascii=False))

{
  "utc": "2026-09-15T11:37:11.280128+00:00",
  "python": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "processor": "x86_64",
  "gpu": null,
  "packages": {
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "torch": "2.11.0+cpu",
    "numba": "0.61.2",
    "scikit-image": "0.25.2",
    "Pillow": "11.3.0",
    "nibabel": "5.4.2",
    "cryptography": "50.0.1"
  },
  "source_sha256": {
    "__init__.py": "ca0d97e51702b28ef3e3814d44d3591e365ae903a24905582728875e60038ea7",
    "__main__.py": "159df03237b3f6dce38f66b78397fbb427921e577061fd90f256ca3821154a6b",
    "artifacts.py": "72329cf074062eb81dcb8d43226a5cb650bbcac67e070b689b7c3ce03cb93ad5",
    "attacks.py": "77dc3e3ebb5d8345353c51bab5df37c9abf7fb99b449c71e608484bbf088aff3",
    "build_delivery.py": "c878097e31cdfacead428bbd40fed0b9bc8aafba2ab03a35d77d43cf52bbe4e8",
    "chaos.py": "4ec83407436b4c08dc7ccbe8f8a4a4e825347fed3bc2a5f673131277ac16142f",
    "cipher.py": "2e657a9c32f9793aa47eebafb550183648b75f708934e6b50

## ۳. مسیرهای ماندگار و تنظیمات
نام/نسخهٔ دقیق داده و مسیر واقعی آن را وارد کنید. مسیر WORK برای وزن و گزارش است؛ DATA_ROOT باید پوشهٔ داده‌های دریافت‌شده با مجوز باشد. برای هر دیتاست جدا اجرا کنید.

In [11]:
"""Paste this whole file in one Colab cell AFTER the Drive/path configuration cell.

Downloads a pinned public REDISTRIBUTION, not an official-author archive.
Does not train, alter source code, fabricate masks, or create a research manifest.
"""
import hashlib
import json
import os
import shutil
import stat
import tempfile
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from urllib.request import Request, urlopen

from PIL import Image

# Public source configuration; terms remain those of the dataset providers.
os.environ['DRIVE_SOURCE_PAGE'] = 'https://www.kaggle.com/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction'
os.environ['DRIVE_OFFICIAL_PAGE'] = 'https://drive.grand-challenge.org/DRIVE/'
os.environ['DRIVE_ARCHIVE_URL'] = 'https://www.kaggle.com/api/v1/datasets/download/andrewmvd/drive-digital-retinal-images-for-vessel-extraction?datasetVersionNumber=1'
EXPECTED_SHA256 = '3efa1bf264da71a9080c9959c5ee89e2646193892efe9d66f29a4123b74f949a'


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def download_archive(destination):
    if not destination.exists():
        partial = destination.with_suffix('.zip.part')
        try:
            request = Request(os.environ['DRIVE_ARCHIVE_URL'], headers={'User-Agent': 'ThesisResearch/1.0'})
            with urlopen(request, timeout=90) as response, partial.open('wb') as f:
                total = 0
                while block := response.read(1024 * 1024):
                    total += len(block)
                    if total > 64 * 1024 * 1024:
                        raise ValueError('حجم پاسخ غیرمنتظره است؛ دریافت متوقف شد.')
                    f.write(block)
                    print(f'دریافت: {total / 1024**2:.1f} MiB', end='\r', flush=True)
            if file_hash(partial) != EXPECTED_SHA256:
                raise ValueError('هش دانلود مطابقت ندارد؛ فایل تغییرکرده، ناقص یا صفحهٔ ورود است.')
            partial.replace(destination)
            print()
        except Exception:
            partial.unlink(missing_ok=True)
            raise
    if file_hash(destination) != EXPECTED_SHA256:
        raise ValueError('آرشیو موجود با نسخهٔ بررسی‌شده متفاوت است؛ بازنویسی نشد.')


def unpack_verified(archive, destination):
    # Validate all members before extraction; never merge into existing user data.
    with zipfile.ZipFile(archive) as z:
        if sum(i.file_size for i in z.infolist()) > 128 * 1024 * 1024:
            raise ValueError('حجم استخراج غیرمنتظره است.')
        for info in z.infolist():
            p = PurePosixPath(info.filename)
            if (p.is_absolute() or '..' in p.parts or '\\' in info.filename
                    or not p.parts or p.parts[0] != 'DRIVE'
                    or stat.S_ISLNK(info.external_attr >> 16)):
                raise ValueError('مسیر غیرمجاز در ZIP')
        if z.testzip() is not None:
            raise ValueError('خرابی CRC آرشیو')
        if destination.exists():
            expected = {str(PurePosixPath(i.filename).relative_to('DRIVE')) for i in z.infolist() if not i.is_dir()}
            present = {str(p.relative_to(destination)) for p in destination.rglob('*') if p.is_file()}
            if expected != present:
                raise ValueError('پوشهٔ DRIVE قبلی متفاوت/ناقص است؛ هیچ فایلی بازنویسی نشد.')
            for info in z.infolist():
                if not info.is_dir():
                    p = destination / PurePosixPath(info.filename).relative_to('DRIVE')
                    if p.is_symlink() or file_hash(p) != hashlib.sha256(z.read(info)).hexdigest():
                        raise ValueError('محتوای DRIVE قبلی متفاوت است؛ بازنویسی نشد: ' + str(p))
        else:
            with tempfile.TemporaryDirectory(prefix='.drive-stage-', dir=destination.parent) as tmp:
                z.extractall(tmp)
                (Path(tmp) / 'DRIVE').rename(destination)


def inventory(root):
    counts = {}
    for split, ids in [('training', range(21, 41)), ('test', range(1, 21))]:
        images = sorted((root / split / 'images').glob('*.tif'))
        if {int(p.stem.split('_')[0]) for p in images} != set(ids) or len(images) != 20:
            raise ValueError('تعداد یا شناسهٔ تصاویر غیرمنتظره: ' + split)
        for path in images:
            with Image.open(path) as im:
                im.load()
                if im.size != (565, 584):
                    raise ValueError('ابعاد غیرمنتظره: ' + str(path))
        masks = sorted((root / split / '1st_manual').glob('*.gif'))
        if split == 'training' and ({int(p.stem.split('_')[0]) for p in masks} != set(ids) or len(masks) != 20):
            raise ValueError('ماسک دستی آموزش ناقص است.')
        for path in masks:
            with Image.open(path) as im:
                im.load()
                if im.size != (565, 584) or set(im.convert('L').tobytes()) != {0, 255}:
                    raise ValueError('ماسک مرجع نامعتبر: ' + str(path))
        counts[split] = {'images': len(images), 'vessel_manual_masks': len(masks)}
    return counts


def main():
    if not os.environ.get('THESIS_WORKDIR'):
        raise RuntimeError('ابتدا سلول ۳، اتصال Drive و تنظیم THESIS_WORKDIR را اجرا کنید.')
    work = Path(os.environ['THESIS_WORKDIR']).resolve()
    mount = Path('/content/drive')
    if not (mount / 'MyDrive').is_dir() or not work.is_relative_to((mount / 'MyDrive').resolve()):
        raise RuntimeError('این گام باید در Drive ماندگار اجرا شود؛ سلول ۳ را با MOUNT_DRIVE=True اجرا کنید.')
    source = work / 'data_sources' / 'DRIVE'
    downloads = work / 'downloads'
    downloads.mkdir(parents=True, exist_ok=True)
    source.parent.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(work).free < 200 * 1024 * 1024:
        raise OSError('فضای آزاد قابل‌مشاهده کافی نیست؛ سهمیهٔ خود Google Drive را نیز بررسی کنید.')
    print('منبع: بازنشر عمومی Kaggle، نسخهٔ ۱؛ نه آرشیو منتشرشده مستقیم توسط مؤلفان.')
    archive = downloads / 'DRIVE-kaggle-v1.zip'
    download_archive(archive)
    unpack_verified(archive, source)
    counts = inventory(source)
    report = {
        'step': '01_download_drive', 'status': 'download_verified',
        'utc': datetime.now(timezone.utc).isoformat(), 'data_root': str(source),
        'source_page': os.environ['DRIVE_SOURCE_PAGE'], 'official_page': os.environ['DRIVE_OFFICIAL_PAGE'],
        'archive_url': os.environ['DRIVE_ARCHIVE_URL'], 'distribution': 'Kaggle redistribution; version 1',
        'sha256': file_hash(archive), 'archive_bytes': archive.stat().st_size, 'counts': counts,
        'checksum_scope': 'Integrity against archive inspected by assistant; NOT independent author authenticity verification',
        'test_vessel_ground_truth_available': counts['test']['vessel_manual_masks'] == 20,
        'warning': 'test/mask is FIELD OF VIEW, NOT vessel ground truth. No test Dice/IoU possible with this archive.',
        'training_executed': False, 'research_manifest_created': False,
    }
    report_path = work / 'drive_download_report.json'
    temporary = report_path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(report_path)
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print('گام ۱ تمام شد. فعلاً سلول ۴ و سلول‌های بعدی قبلی را اجرا نکنید.')
    return report


if __name__ == '__main__':
    STEP1_REPORT = main()

منبع: بازنشر عمومی Kaggle، نسخهٔ ۱؛ نه آرشیو منتشرشده مستقیم توسط مؤلفان.
{
  "step": "01_download_drive",
  "status": "download_verified",
  "utc": "2026-09-15T12:23:11.211524+00:00",
  "data_root": "/content/drive/MyDrive/Thesis_Research/data_sources/DRIVE",
  "source_page": "https://www.kaggle.com/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction",
  "official_page": "https://drive.grand-challenge.org/DRIVE/",
  "archive_url": "https://www.kaggle.com/api/v1/datasets/download/andrewmvd/drive-digital-retinal-images-for-vessel-extraction?datasetVersionNumber=1",
  "distribution": "Kaggle redistribution; version 1",
  "sha256": "3efa1bf264da71a9080c9959c5ee89e2646193892efe9d66f29a4123b74f949a",
  "archive_bytes": 29351684,
  "counts": {
    "training": {
      "images": 20,
      "vessel_manual_masks": 20
    },
    "test": {
      "images": 20,
      "vessel_manual_masks": 0
    }
  },
  "checksum_scope": "Integrity against archive inspected by assistant; NOT indepe

In [16]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from importlib.metadata import version
import os
import json
import hashlib

if "WORK" not in globals():
    raise RuntimeError("ابتدا سلول ۳، تنظیمات مسیرها، را اجرا کنید.")

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("ابتدا Google Drive را متصل کنید.")

work = Path(WORK)
cache_root = work / "data_sources" / "_kaggle_cache"
os.environ["KAGGLEHUB_CACHE"] = str(cache_root)

import kagglehub

catalog = {
    "CHASE_DB1": "namnguynnnn/chase-db1/versions/1",
    "STARE": "aryankamani/stare-dataset-20images/versions/1",
    "FIVES": (
        "nikitamanaenkov/"
        "fundus-image-dataset-for-vessel-segmentation/versions/1"
    ),
}

# فعلاً دو مجموعهٔ کوچک‌تر؛ FIVES در این اجرا دانلود نمی‌شود.
selected_datasets = ["CHASE_DB1", "STARE"]

receipt_dir = work / "data_provenance"
receipt_dir.mkdir(parents=True, exist_ok=True)
EXTRA_DATA_ROOTS = {}

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for name in selected_datasets:
    handle = catalog[name]
    print(f"\nدریافت {name}: {handle}", flush=True)

    # خطای دانلود متوقف‌کننده است؛ دادهٔ جایگزین تولید نمی‌شود.
    root = Path(kagglehub.dataset_download(handle)).resolve()

    if not root.is_relative_to(cache_root.resolve()):
        raise RuntimeError(
            f"مسیر دریافت خارج از محل ماندگار مورد انتظار است: {root}"
        )

    downloaded_files = sorted(
        p for p in root.rglob("*") if p.is_file()
    )
    if not downloaded_files:
        raise RuntimeError(f"هیچ فایلی برای {name} دریافت نشد.")

    inventory = []
    folders = defaultdict(list)

    for path in downloaded_files:
        relative = path.relative_to(root)
        folders[str(relative.parent)].append(path.name)
        inventory.append({
            "path": str(relative),
            "bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        })

    now = datetime.now(timezone.utc)
    receipt = {
        "dataset": name,
        "handle": handle,
        "kaggle_url": f"https://www.kaggle.com/datasets/{handle}",
        "local_root": str(root),
        "checked_at_utc": now.isoformat(),
        "kagglehub_version": version("kagglehub"),
        "status": "downloaded_pending_annotation_validation",
        "files": inventory,
    }

    receipt_path = receipt_dir / (
        f"{name}_{now.strftime('%Y%m%dT%H%M%S%fZ')}.json"
    )
    with receipt_path.open("x", encoding="utf-8") as stream:
        json.dump(receipt, stream, ensure_ascii=False, indent=2)

    EXTRA_DATA_ROOTS[name] = root
    print("مسیر:", root)
    print("تعداد کل فایل‌ها:", len(downloaded_files))
    print("رسید:", receipt_path)

    for folder, names in sorted(folders.items())[:15]:
        print(f"  {folder}: {len(names)} فایل")
        print("    نمونه:", names[:3])

print("\nدانلود پایان یافت؛ اعتبار برچسب‌ها هنوز بررسی نشده است.")


دریافت CHASE_DB1: namnguynnnn/chase-db1/versions/1


100%|██████████| 2.04M/2.04M [00:00<00:00, 70.3MB/s]

Extracting files...


مسیر: /content/drive/MyDrive/Thesis_Research/data_sources/_kaggle_cache/datasets/namnguynnnn/chase-db1/versions/1
تعداد کل فایل‌ها: 56
رسید: /content/drive/MyDrive/Thesis_Research/data_provenance/CHASE_DB1_20260915T131828423897Z.json
  images: 28 فایل
    نمونه: ['Image_01L.jpg', 'Image_01R.jpg', 'Image_02L.jpg']
  masks: 28 فایل
    نمونه: ['Image_01L_1stHO.png', 'Image_01R_1stHO.png', 'Image_02L_1stHO.png']

دریافت STARE: aryankamani/stare-dataset-20images/versions/1


100%|██████████| 18.0M/18.0M [00:00<00:00, 53.4MB/s]

Extracting files...


مسیر: /content/drive/MyDrive/Thesis_Research/data_sources/_kaggle_cache/datasets/aryankamani/stare-dataset-20images/versions/1
تعداد کل فایل‌ها: 40
رسید: /content/drive/MyDrive/Thesis_Research/data_provenance/STARE_20260915T131832251485Z.json
  labels-ah: 20 فایل
    نمونه: ['im0001.ah.ppm', 'im0002.ah.ppm', 'im0003.ah.ppm']
  stare-images: 20 فایل
    نمونه: ['im0001.ppm', 'im0002.ppm', 'im0003.ppm']

دانلود پایان یافت؛ اعتبار برچسب‌ها هنوز بررسی نشده است.


In [17]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from importlib.metadata import version
import os
import json
import hashlib

if "WORK" not in globals():
    raise RuntimeError("ابتدا سلول ۳، تنظیمات مسیرها، را اجرا کنید.")

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("ابتدا Google Drive را متصل کنید.")

work = Path(WORK)
cache_root = work / "data_sources" / "_kaggle_cache"
os.environ["KAGGLEHUB_CACHE"] = str(cache_root)

import kagglehub

catalog = {
    "CHASE_DB1": "namnguynnnn/chase-db1/versions/1",
    "STARE": "aryankamani/stare-dataset-20images/versions/1",
    "FIVES": (
        "nikitamanaenkov/"
        "fundus-image-dataset-for-vessel-segmentation/versions/1"
    ),
}

# فعلاً دو مجموعهٔ کوچک‌تر؛ FIVES در این اجرا دانلود نمی‌شود.
selected_datasets = ["FIVES"]

receipt_dir = work / "data_provenance"
receipt_dir.mkdir(parents=True, exist_ok=True)
EXTRA_DATA_ROOTS = {}

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for name in selected_datasets:
    handle = catalog[name]
    print(f"\nدریافت {name}: {handle}", flush=True)

    # خطای دانلود متوقف‌کننده است؛ دادهٔ جایگزین تولید نمی‌شود.
    root = Path(kagglehub.dataset_download(handle)).resolve()

    if not root.is_relative_to(cache_root.resolve()):
        raise RuntimeError(
            f"مسیر دریافت خارج از محل ماندگار مورد انتظار است: {root}"
        )

    downloaded_files = sorted(
        p for p in root.rglob("*") if p.is_file()
    )
    if not downloaded_files:
        raise RuntimeError(f"هیچ فایلی برای {name} دریافت نشد.")

    inventory = []
    folders = defaultdict(list)

    for path in downloaded_files:
        relative = path.relative_to(root)
        folders[str(relative.parent)].append(path.name)
        inventory.append({
            "path": str(relative),
            "bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        })

    now = datetime.now(timezone.utc)
    receipt = {
        "dataset": name,
        "handle": handle,
        "kaggle_url": f"https://www.kaggle.com/datasets/{handle}",
        "local_root": str(root),
        "checked_at_utc": now.isoformat(),
        "kagglehub_version": version("kagglehub"),
        "status": "downloaded_pending_annotation_validation",
        "files": inventory,
    }

    receipt_path = receipt_dir / (
        f"{name}_{now.strftime('%Y%m%dT%H%M%S%fZ')}.json"
    )
    with receipt_path.open("x", encoding="utf-8") as stream:
        json.dump(receipt, stream, ensure_ascii=False, indent=2)

    EXTRA_DATA_ROOTS[name] = root
    print("مسیر:", root)
    print("تعداد کل فایل‌ها:", len(downloaded_files))
    print("رسید:", receipt_path)

    for folder, names in sorted(folders.items())[:15]:
        print(f"  {folder}: {len(names)} فایل")
        print("    نمونه:", names[:3])

print("\nدانلود پایان یافت؛ اعتبار برچسب‌ها هنوز بررسی نشده است.")


دریافت FIVES: nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation/versions/1


KaggleApiHTTPError: 404 Client Error.

Resource not found at URL: https://kaggle.com/datasets/nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation/versions/1
Please make sure you specified the correct resource identifiers.

In [3]:
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
os.environ['THESIS_WORKDIR'] = '/content/drive/MyDrive/Thesis_Research' if MOUNT_DRIVE else '/content/Thesis_Research'
WORK = Path(os.environ['THESIS_WORKDIR']); WORK.mkdir(parents=True, exist_ok=True)
DATASET = 'DRIVE'  # DRIVE / RITE / BraTS2020 / COVID19_CXR
DATA_ROOT = WORK/'data_sources'/DATASET
SOURCE = ''  # شناسه/نسخه و منشأ واقعی داده؛ خالی مجاز نیست
CXR_CSV = WORK/'cxr_index.csv'
MANIFEST = WORK/f'{DATASET}.jsonl'
WEIGHTS = WORK/'weights'/DATASET
PAYLOAD = WORK/'metadata.bin'  # فایل آزمایشی فاقد اطلاعات هویتی بیمار
RUN_DIR = WORK/'runs'/f'{DATASET}_001'  # در اجرای جدید نام تازه انتخاب کنید
PROFILE = PACKAGE/'scripts/thesis_reference.json'  # نسخه مستند تکمیل‌شده خودتان را جایگزین کنید
RUN_TRAINING = False
RESUME_TRAINING = False
RUN_EXPERIMENTS = False
RUN_NIST = False
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

Mounted at /content/drive
device: cpu


این کد فقط وضعیت مسیرها را نمایش می‌دهد؛ چیزی دانلود، ایجاد یا تغییر نمی‌دهد:

In [12]:
from pathlib import Path

required = ["WORK", "DATASET", "DATA_ROOT", "SOURCE", "MANIFEST"]
missing = [name for name in required if name not in globals()]

if missing:
    print("ابتدا سلول تنظیمات، یعنی بخش ۳، را اجرا کنید.")
    print("متغیرهای تعریف‌نشده:", missing)
else:
    root = Path(DATA_ROOT)
    manifest = Path(MANIFEST)

    print("DATASET:", DATASET)
    print("WORK:", WORK)
    print("DATA_ROOT:", root)
    print("پوشهٔ داده موجود است؟", root.is_dir())
    print("SOURCE:", repr(SOURCE))
    print("MANIFEST:", manifest)
    print("فایل فهرست موجود است؟", manifest.is_file())

    if root.is_dir():
        entries = sorted(root.iterdir(), key=lambda p: p.name)
        print("\nتعداد موارد مستقیم داخل پوشه:", len(entries))
        print("حداکثر ۲۰ مورد اول:")
        for item in entries[:20]:
            kind = "پوشه" if item.is_dir() else "فایل"
            print(f"  [{kind}] {item.name}")
    else:
        parent = root.parent
        print("\nپوشهٔ والد موجود است؟", parent.is_dir())
        if parent.is_dir():
            print("پوشه‌های موجود در والد:")
            for item in sorted(parent.iterdir(), key=lambda p: p.name):
                if item.is_dir():
                    print(" ", item.name)

DATASET: DRIVE
WORK: /content/drive/MyDrive/Thesis_Research
DATA_ROOT: /content/drive/MyDrive/Thesis_Research/data_sources/DRIVE
پوشهٔ داده موجود است؟ True
SOURCE: ''
MANIFEST: /content/drive/MyDrive/Thesis_Research/DRIVE.jsonl
فایل فهرست موجود است؟ False

تعداد موارد مستقیم داخل پوشه: 2
حداکثر ۲۰ مورد اول:
  [پوشه] test
  [پوشه] training


In [13]:
from pathlib import Path

root = Path(DATA_ROOT)

for split in ("training", "test"):
    split_dir = root / split
    print(f"\n--- {split} ---")

    if not split_dir.is_dir():
        print("پوشه موجود نیست:", split_dir)
        continue

    folders = sorted(
        (p for p in split_dir.iterdir() if p.is_dir()),
        key=lambda p: p.name
    )

    if not folders:
        print("هیچ زیرپوشه‌ای پیدا نشد.")

    for folder in folders:
        items = sorted(
            (p for p in folder.iterdir()
             if p.is_file() and not p.name.startswith(".")),
            key=lambda p: p.name
        )
        print(f"{folder.name}: {len(items)} فایل")
        print("  نمونه نام‌ها:", [p.name for p in items[:3]])


--- training ---
1st_manual: 20 فایل
  نمونه نام‌ها: ['21_manual1.gif', '22_manual1.gif', '23_manual1.gif']
images: 20 فایل
  نمونه نام‌ها: ['21_training.tif', '22_training.tif', '23_training.tif']
mask: 20 فایل
  نمونه نام‌ها: ['21_training_mask.gif', '22_training_mask.gif', '23_training_mask.gif']

--- test ---
images: 20 فایل
  نمونه نام‌ها: ['01_test.tif', '02_test.tif', '03_test.tif']
mask: 20 فایل
  نمونه نام‌ها: ['01_test_mask.gif', '02_test_mask.gif', '03_test_mask.gif']


In [14]:
import inspect
from scripts.prepare import retinal
from scripts.data import load_manifest

for function in (retinal, load_manifest):
    print("\n" + "=" * 70)
    print(f"FUNCTION: {function.__module__}.{function.__name__}")
    print("=" * 70)
    print(inspect.getsource(function))


FUNCTION: scripts.prepare.retinal
def retinal(root, dataset, source):
    records = []
    for official in ('training', 'test'):
        folder = root / official
        images = sorted((folder / 'images').glob('*'))
        images = [p for p in images if p.suffix.lower() in ('.tif', '.tiff', '.png', '.jpg')]
        for image in images:
            ident = image.stem.split('_')[0]
            if not ident.isdigit(): raise ValueError(f'Cannot infer original DRIVE/RITE ID: {image}')
            ident = f'{int(ident):02d}'
            masks = [p for p in folder.rglob('*') if p.is_file() and p.stem.split('_')[0] == ident
                     and (('1st_manual' in str(p)) if dataset == 'DRIVE' else ('av' in p.parent.name.lower()))]
            if len(masks) != 1: raise ValueError(f'Need exactly one first-observer/AV mask for {image}: {masks}')
            # Official test untouched; same four train IDs held out in both retinal datasets.
            split = 'test' if official == 'test' else

## ۴. ساخت فهرست دادهٔ واقعی
اگر ساختار شما متفاوت است، مطابق README فایل JSONL صریح تهیه کنید. این سلول هیچ داده‌ای دانلود یا جعل نمی‌کند. برای BraTS اسلایس‌ها بعد از تقسیم بیمار ساخته می‌شوند؛ DRIVE/RITE از گروه‌های مشترک استفاده می‌کنند.

In [4]:
from scripts.prepare import prepare
from scripts.data import load_manifest, load_sample
if not MANIFEST.exists():
    prepare(DATASET, DATA_ROOT, MANIFEST, SOURCE, csv_path=CXR_CSV if DATASET == 'COVID19_CXR' else None,
            modalities=['flair', 't1', 't1ce', 't2'], stride=1)
records = load_manifest(MANIFEST)
from collections import Counter
print(Counter((r['dataset'], r['split']) for r in records))
example, reference_mask, prep = load_sample(records[0])
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(example, cmap='gray'); axes[0].set_title('Real processed input')
axes[1].imshow(reference_mask, cmap='gray'); axes[1].set_title('Reference ROI')
plt.show(); print(prep)

ValueError: Existing root and documented source required

## ۵. آموزش و ارزیابی U-Net
آموزش به عددهای ODE نیاز ندارد. `RUN_TRAINING=True` را پس از بررسی داده فعال کنید؛ دوره‌ها و دیگر تنظیمات در کپی JSON قابل تغییرند. بهترین وزن فقط از اعتبارسنجی انتخاب می‌شود. با قطع نشست و همان تنظیمات، RESUME_TRAINING را فعال کنید.

In [5]:
from scripts.training import train
from scripts.experiments import evaluate_segmentation
cfg = json.loads(PROFILE.read_text())
if RUN_TRAINING:
    print(train(MANIFEST, DATASET, cfg, WEIGHTS, DEVICE, resume=RESUME_TRAINING))
CHECKPOINT = WEIGHTS/'best.pt'
if CHECKPOINT.exists():
    evaluation = evaluate_segmentation(MANIFEST, DATASET, CHECKPOINT, WORK/'evaluation'/f'{DATASET}.json', DEVICE)
    print({k:v for k,v in evaluation.items() if k not in ('records','environment')})
else:
    print('وزن آموزش‌دیده موجود نیست؛ ارزیابی یا استنتاج با وزن تصادفی انجام نشد.')

وزن آموزش‌دیده موجود نیست؛ ارزیابی یا استنتاج با وزن تصادفی انجام نشد.


## ۶. اعتبار روش پیش از رمزنگاری
تنظیمات معادلهٔ ۳–۲ عمداً ناقص‌اند. پروفایل پیوست جایگزین بی‌نام نیست. برای استفادهٔ آزمایشی از آن باید PROFILE را صریحاً تغییر دهید و نتیجه را تفسیر پیوست بنامید. عددهای مطلوب دلیل انتخاب پارامتر نیستند.

In [6]:
from scripts.config import validate
from scripts.model import Predictor
from scripts.chaos import roi_digest, stream
from scripts.artifacts import write_json
METHOD_READY = False
try:
    validate(cfg)
    predictor = Predictor(CHECKPOINT, DEVICE)
    selected = next(r for r in records if r['split'] == 'test')
    image, target, preprocessing = load_sample(selected)
    predicted_roi = predictor(image)
    digest, moments = roi_digest(image, predicted_roi)
    raw = stream(digest, image.size, cfg, raw=True)
    METHOD_READY = True
    print('پیش‌بررسی این نمونه موفق؛ هنوز اثبات پایداری یا امنیت نیست.', moments)
except Exception as exc:
    write_json(WORK/'method_preflight.json', {'status':'blocked','error':f'{type(exc).__name__}: {exc}','config':cfg})
    print('مانع ثبت‌شده:', exc)

مانع ثبت‌شده: پارامترهای رابطه ۳-۲ در پایان‌نامه عدد ندارند؛ ابتدا همه را مستند و تعیین کنید. پروفایل پیوست روش متفاوتی است.


## ۷. اجرای آزمایش‌ها
برای هر چهار مجموعه جدا اجرا کنید. پیش‌فرض ۱۰۰ تلاش تفاضلی در هر مجموعه است؛ نویز/برش، حساسیت digest، حذف مؤلفه‌ها و بازیابی دقیق هم ثبت می‌شوند. طول پیام و ظرفیت واقعی‌اند. اجرای همهٔ مجموعه‌ها ممکن است از زمان یک نشست کولب طولانی‌تر باشد.

In [7]:
from scripts.experiments import run
from scripts.reporting import make_report
if RUN_EXPERIMENTS:
    if not METHOD_READY: raise RuntimeError('ابتدا مانع مشخصات/وزن/دینامیک را رفع و مستند کنید')
    if not PAYLOAD.is_file(): raise FileNotFoundError('فایل فراداده آزمایشی لازم است: '+str(PAYLOAD))
    status = run(MANIFEST, DATASET, CHECKPOINT, cfg, PAYLOAD, RUN_DIR, DEVICE, include_ablations=True)
    print(status)
    print(make_report(RUN_DIR))
else:
    print('اجرای پژوهشی فعال نشده است؛ نتیجه‌ای ساخته نشد.')

اجرای پژوهشی فعال نشده است؛ نتیجه‌ای ساخته نشد.


## ۸. رفت‌وبرگشت مستقل تصویر و پیام
فایل راز خارج از گزارش و فایل‌های ارسالی نگه‌داری می‌شود. گیرنده به stego، recovery.msr و همان راز نیاز دارد. این افزونهٔ اصلاحی با ادعای گیرندهٔ بدون فایل کمکی پایان‌نامه متفاوت است.

In [8]:
def cli(*args):
    return subprocess.run([sys.executable, '-m', 'scripts', *map(str,args)], check=True)
RUN_DELIVERY = False
if RUN_DELIVERY:
    if not METHOD_READY: raise RuntimeError('روش آماده نیست')
    KEYFILE = WORK/'private_research.key'
    if not KEYFILE.exists(): cli('keygen','--output',KEYFILE)
    SENT = WORK/'sent_001'; RECEIVED = WORK/'received_001'
    cli('send','--manifest',MANIFEST,'--dataset',DATASET,'--id',selected['id'],'--checkpoint',CHECKPOINT,
        '--config',PROFILE,'--payload',PAYLOAD,'--keyfile',KEYFILE,'--output',SENT,'--device',DEVICE)
    cli('receive','--stego',SENT/'stego.png','--sidecar',SENT/'recovery.msr','--keyfile',KEYFILE,'--output',RECEIVED)

## ۹. برآورد لیاپانوف و پرترهٔ فاز
این برآورد زمان محدود است. طول‌های متفاوت، چند شرایط اولیه و گام‌های کوچکتر برای بررسی همگرایی لازم‌اند؛ علامت مثبت به‌تنهایی اثبات نیست.

In [9]:
from scripts.dynamics import lyapunov
if METHOD_READY and RUN_EXPERIMENTS:
    dynamics = lyapunov(digest, cfg, steps=100000, qr_interval=10, output=WORK/'dynamics'/f'{DATASET}.json')
    print({k:v for k,v in dynamics.items() if k not in ('phase','convergence','environment')})
    if dynamics['status'] == 'ok':
        import numpy as np
        phase = np.array(dynamics['phase'])
        fig = plt.figure(); ax = fig.add_subplot(111, projection='3d')
        ax.plot(phase[:,0], phase[:,1], phase[:,2], linewidth=.4)
        ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z'); plt.show()

## ۱۰. NIST رسمی: دریافت، ساخت و اجرا
این مرحله بدون دادهٔ کافی متوقف می‌شود؛ هیچ پرکردن/تکرار تصویر برای رسیدن به صد میلیون بیت انجام نمی‌شود. خروجی تمام ۱۵ خانواده و مؤلفه‌های آن‌ها در پوشهٔ نتیجه باقی می‌ماند. دادهٔ کمتر با برچسب آزمون نرم‌افزار از این مرحلهٔ پژوهشی جدا است.

In [ ]:
os.environ['NIST_STS_URL'] = 'https://csrc.nist.gov/CSRC/media/Projects/Random-Bit-Generation/documents/sts-2_1_2.zip'
NIST_RUNS = [WORK/'runs'/f'{name}_001' for name in ['DRIVE','RITE','BraTS2020','COVID19_CXR']]
if RUN_NIST:
    import urllib.request
    from scripts.nist import build_official, export_streams, run_official
    archive = WORK/'sts-2_1_2.zip'
    if not archive.exists(): urllib.request.urlretrieve(os.environ['NIST_STS_URL'], archive)
    BUILD = WORK/'nist_build'
    if not BUILD.exists(): sts_root = build_official(archive, BUILD)
    else: sts_root = Path(json.loads((BUILD/'build.json').read_text())['assess']).parent
    nist_input = WORK/'nist_input_001'
    if not nist_input.exists(): export_streams(NIST_RUNS, nist_input, sequences=100, bits=1000000)
    nist_result = run_official(sts_root, nist_input, WORK/'nist_results_001', timeout=7200)
    print(nist_result['status'], 'families:', nist_result['families'])

## ۱۱. جمع‌بندی و دریافت جدول‌ها
فقط گزارش اجرای انتخابی بسته‌بندی می‌شود؛ فایل راز و دادهٔ خام پزشکی در ZIP گزارش نیست. جدول literature_reported.csv نقل از پایان‌نامه است و بازاجرای مقاله‌ها نیست. زمان کولب را زمان Jetson معرفی نکنید.

In [ ]:
if (RUN_DIR/'status.json').exists():
    make_report(RUN_DIR)
    import shutil
    archive = shutil.make_archive(str(WORK/f'{DATASET}_report'), 'zip', RUN_DIR/'reports')
    files.download(archive)
else:
    print('هنوز اجرای ثبت‌شده‌ای برای گزارش وجود ندارد.')